# 🌟 Exercise 1: Setup and Environment Configuration

## Objectives
- Install LangChain library
- Sign up for LangSmith and configure tracing
- Set up Tavily search tool API
- Verify environment configuration

In [ ]:
# Install LangChain
!pip install langchain

In [ ]:

import langchain
print(f"LangChain version: {langchain.__version__}")

## Step 1.3: Enable Tracing with LangSmith

**Instructions:**
1. Sign up for LangSmith at: https://smith.langchain.com/
2. Get your API key from the settings
3. Run the cell below and enter your API key when prompted

In [ ]:
import getpass
import os

# Enable LangSmith tracing
os.environ["LANGSMITH_TRACING"] = "true"

# Securely input your LangSmith API key
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass(prompt="LangSmith API Key: ")
    
print("LangSmith tracing enabled!")

## Step 1.4: Configure Tavily API Key

**Instructions:**
1. Register at Tavily's developer portal: https://tavily.com/
2. Obtain your TAVILY_API_KEY
3. Run the cell below and enter your API key when prompted

In [ ]:
import getpass
import os

# Securely input your Tavily API key
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass(prompt="Tavily API Key: ")
    
print("Tavily API key configured!")

## Step 1.5: Verify Your Configuration

In [ ]:
import os

# Confirm that tracing is active and the Tavily key is set
assert os.getenv("LANGSMITH_TRACING") == "true", "LangSmith tracing not enabled"
assert os.getenv("LANGSMITH_API_KEY") is not None, "LangSmith API key not set"
assert os.getenv("TAVILY_API_KEY") is not None, "Tavily API key not set"

print("✅ Environment configured successfully!")
print(f"   - LangSmith Tracing: {os.getenv('LANGSMITH_TRACING')}")
print(f"   - LangSmith API Key: {'*' * 20} (hidden)")
print(f"   - Tavily API Key: {'*' * 20} (hidden)")

# 🌟 Exercise 2: Define Tools

## Objectives
- Install LangChain community tools
- Import and instantiate TavilySearchResults
- Test the search tool
- Create a tools list for agent use

## Step 2.1: Install LangChain Community Tools

In [ ]:
# Install LangChain and community tools
!pip install langchain langchain-community

## Step 2.2: Import the Tavily Search Tool

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

print("✅ TavilySearchResults imported successfully!")

## Step 2.3: Instantiate the Search Tool

In [ ]:
# Create the search object (it will read TAVILY_API_KEY from the environment)
search = TavilySearchResults(max_results=5)

print("✅ Search tool instantiated successfully!")

## Step 2.4: Run a Test Query

In [ ]:
# Use the search.invoke method to perform a query
search_results = search.invoke("latest advancements in AI-driven healthcare triage")

print("Search results:")
print("-" * 80)
for i, result in enumerate(search_results, 1):
    print(f"\n{i}. {result.get('title', 'No title')}")
    print(f"   URL: {result.get('url', 'No URL')}")
    print(f"   Snippet: {result.get('content', 'No content')[:200]}...")

## Step 2.5: Assemble Your Tools List

In [ ]:
# Add your search tool to a list for use in chains or agents
tools = [search]

print(f"✅ Tools list created with {len(tools)} tool(s):")
for tool in tools:
    print(f"   - {tool.name}: {tool.description}")

# 🌟 Exercise 3: Using Language Models

## Objectives
- Install Mistral integration
- Configure Mistral API key
- Initialize a chat model
- Send messages and receive responses

## Step 3.1: Install the Mistral Integration

In [ ]:
# Install LangChain with Mistral support
!pip install -qU langchain-mistralai

## Step 3.2: Set Your Mistral API Key

**Instructions:**
1. Sign up at: https://console.mistral.ai/
2. Get your API key
3. Run the cell below and enter your API key when prompted

In [ ]:
import getpass
import os

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Enter API key for Mistral AI: ")

print("✅ Mistral API key configured!")

## Step 3.3: Import and Initialize the Chat Model

In [ ]:
from langchain_mistralai import ChatMistralAI

# Create your model instance
model = ChatMistralAI(
    model="mistral-small-latest",  # or "mistral-medium-latest", "mistral-large-latest"
    api_key=os.environ["MISTRAL_API_KEY"],
    temperature=0.7
)

print("✅ Chat model initialized successfully!")
print(f"   Model: {model.model}")

## Step 3.4: Prepare a HumanMessage

In [ ]:
from langchain_core.messages import HumanMessage

# Construct a prompt
msg = HumanMessage(content="Tell me a fun fact about space exploration.")

print("✅ Message prepared!")

## Step 3.5: Call the Model and Retrieve the Response

In [ ]:
# Pass your message in a list and print the returned content
response = model.invoke([msg])

print("Response from Mistral:")
print("-" * 80)
print(response.content)

# 🌟 Exercise 4: Tool Binding and Response Inspection

## Objectives
- Bind tools to the language model
- Test normal model calls without tool invocation
- Test calls that trigger tool usage
- Inspect response attributes (.content and .tool_calls)

## Step 4.1: Bind Your Tools to the Model

In [ ]:
# Use .bind_tools to create a new model instance that knows about your tools
model_with_tools = model.bind_tools(tools)

print("✅ Tools bound to model successfully!")
print(f"   Model now has access to {len(tools)} tool(s)")

## Step 4.2: Call the Model Normally (Without Tool)

In [ ]:
# Send a simple prompt that does not require a tool
response = model_with_tools.invoke([HumanMessage(content="What's 2 + 2?")])

print("Response for simple query:")
print("-" * 80)
print(f"Content: {response.content}")
print(f"Tool Calls: {response.tool_calls}")
print("\n✅ As expected, no tools were called for this simple math question.")

## Step 4.3: Call the Model Expecting a Tool Invocation

In [ ]:
# Send a query that requires external information (e.g., a web search)
response = model_with_tools.invoke([
    HumanMessage(content="Search for the latest AI breakthroughs in healthcare.")
])

print("Response for search query:")
print("-" * 80)
print(f"Content: {response.content}")
print(f"\nTool Calls: {response.tool_calls}")

if response.tool_calls:
    print("\n✅ Model correctly decided to use a tool!")
    for tool_call in response.tool_calls:
        print(f"   - Tool: {tool_call.get('name', 'unknown')}")
        print(f"   - Arguments: {tool_call.get('args', {})}")
else:
    print("\n⚠️ Model did not invoke a tool. This might happen sometimes.")

## Step 4.4: Interpret the Output

In [ ]:
# Confirm that the model is correctly generating a tool_call rather than plain text
print("Summary:")
print("-" * 80)
if response.tool_calls:
    print("✅ The model correctly identified that a search tool should be used.")
    print("   The tool_calls attribute contains the function name and arguments.")
    print("   This information can be passed to an agent executor for actual tool execution.")
else:
    print("   The model chose to answer directly without using tools.")

# 🌟 Exercise 5: Create the Agent

## Objectives
- Import LangGraph's React agent builder
- Initialize an agent executor with LLM and tools
- Verify the agent is ready

## Step 5.1: Install LangGraph

In [ ]:
# Install LangGraph
!pip install langgraph

## Step 5.2: Import the React Agent Builder

In [ ]:
from langgraph.prebuilt import create_react_agent

print("✅ create_react_agent imported successfully!")

## Step 5.3: Initialize the Agent Executor

In [ ]:
# Call create_react_agent, passing your base model and tools list
agent_executor = create_react_agent(
    model=model,
    tools=tools
)

print("✅ Agent executor created successfully!")

## Step 5.4: Verify the Agent

In [ ]:
# Check that the agent is ready
print("Agent executor details:")
print("-" * 80)
print(f"Type: {type(agent_executor)}")
print(f"Available tools: {[tool.name for tool in tools]}")
print(f"\n✅ Agent is ready to process queries!")

# 🌟 Exercise 6: Run the Agent

## Objectives
- Run stateless queries through the agent
- Inspect the final state dictionary
- Invoke the agent with queries that trigger tool usage
- Verify tool invocations in the response

## Step 6.1: Run Stateless Queries

In [ ]:
# Example 1: Simple question
print("Query 1: What is the capital of France?")
print("-" * 80)

state1 = agent_executor.invoke({"messages": [HumanMessage(content="What is the capital of France?")]})

# Get the final response
final_messages = state1["messages"]
print(f"Response: {final_messages[-1].content}")
print()

In [ ]:
# Example 2: Request a joke
print("Query 2: Tell me a joke about robots.")
print("-" * 80)

state2 = agent_executor.invoke({"messages": [HumanMessage(content="Tell me a joke about robots.")]})

# Get the final response
final_messages = state2["messages"]
print(f"Response: {final_messages[-1].content}")

## Step 6.2: Inspect the Final State

In [ ]:
# Inspect the structure of the state
print("State structure:")
print("-" * 80)
print(f"Keys in state: {list(state1.keys())}")
print(f"\nNumber of messages: {len(state1['messages'])}")
print(f"Message types: {[type(msg).__name__ for msg in state1['messages']]}")

## Step 6.3: Invoke the Tool via the Agent

In [ ]:
# Send a prompt that should trigger the Tavily search tool
print("Query 3: Find the latest breakthroughs in AI for emergency medicine.")
print("-" * 80)

search_state = agent_executor.invoke({
    "messages": [HumanMessage(content="Find the latest breakthroughs in AI for emergency medicine.")]
})

# Display all messages in the conversation
print("Conversation flow:")
for i, msg in enumerate(search_state["messages"], 1):
    msg_type = type(msg).__name__
    print(f"\n{i}. {msg_type}:")
    
    if hasattr(msg, 'content') and msg.content:
        print(f"   Content: {msg.content[:200]}...")
    
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"   Tool Calls: {msg.tool_calls}")

## Step 6.4: Verify Tool Invocation

In [ ]:
# Analyze the agent's decision-making
print("Tool Invocation Analysis:")
print("-" * 80)

tool_used = False
for msg in search_state["messages"]:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        tool_used = True
        print("✅ Agent decided to use a tool!")
        for tool_call in msg.tool_calls:
            print(f"\n   Tool name: {tool_call.get('name', 'unknown')}")
            print(f"   Arguments: {tool_call.get('args', {})}")

if tool_used:
    print("\n✅ The agent successfully invoked the search tool to answer the query!")
else:
    print("\n⚠️ No tool was invoked. The model might have answered from its training data.")

# Display final answer
print(f"\nFinal Answer:")
print("-" * 80)
print(search_state["messages"][-1].content)

# 🌟 Exercise 7: Streaming Messages

## Objectives
- Use the agent's streaming capability
- Process incremental responses as they arrive
- Pretty-print streaming output

## Step 7.1: Prepare Your Prompt

In [ ]:
from langchain_core.messages import HumanMessage

# Create a list with a single HumanMessage containing your query
messages = [HumanMessage(content="Find the top 3 recent AI papers on medical imaging.")]

print("✅ Streaming prompt prepared!")

## Step 7.2: Call .stream on the Agent Executor

In [ ]:
# Stream the agent's response
print("Streaming Agent Response:")
print("=" * 80)

stream_iterator = agent_executor.stream(
    {"messages": messages},
    stream_mode="values"  # "values" gives us the full state at each step
)

# Iterate and display each step
for step in stream_iterator:
    # Get the latest message
    latest_message = step["messages"][-1]
    msg_type = type(latest_message).__name__
    
    print(f"\n[{msg_type}]")
    print("-" * 80)
    
    if hasattr(latest_message, 'content') and latest_message.content:
        print(latest_message.content)
    
    if hasattr(latest_message, 'tool_calls') and latest_message.tool_calls:
        print(f"Tool Calls: {latest_message.tool_calls}")

print("\n" + "=" * 80)
print("✅ Streaming complete!")

## Step 7.3: Alternative - Using pretty_print()

In [ ]:
# Alternative approach using pretty_print
messages = [HumanMessage(content="What are the benefits of using LangChain for AI development?")]

print("Streaming with pretty_print:")
print("=" * 80)

for step in agent_executor.stream({"messages": messages}, stream_mode="values"):
    step["messages"][-1].pretty_print()

print("\n✅ Streaming complete!")

# 🎉 Congratulations!

You have successfully completed all the LangChain exercises:

1. ✅ **Setup and Environment Configuration** - Configured LangChain, LangSmith, and Tavily
2. ✅ **Define Tools** - Created and tested the Tavily search tool
3. ✅ **Using Language Models** - Initialized and used Mistral AI
4. ✅ **Tool Binding and Response Inspection** - Bound tools to the model and analyzed responses
5. ✅ **Create the Agent** - Built a ReAct-style agent with LangGraph
6. ✅ **Run the Agent** - Executed queries and verified tool invocations
7. ✅ **Streaming Messages** - Implemented streaming responses

## What You Built
- ✨ A working LangChain environment with LangSmith tracing
- 🔍 A search-enabled intelligent agent using Tavily and Mistral
- 🤖 A ReAct-style agent that can reason and act based on inputs
- 💬 A conversational AI system with tool invocation and streaming capabilities

## Next Steps
- Experiment with different models and tools
- Build more complex multi-step workflows
- Explore LangSmith tracing to debug your agents
- Create custom tools for specific use cases